# 03 — Feature Engineering

Extract time-domain and frequency-domain features from preprocessed EEG epochs and analyse their discriminative power.

**Topics:**
- Time-domain features (mean, RMS, Hjorth parameters, ZCR…)
- Frequency-domain features (band power, spectral entropy…)
- Feature correlation matrix
- t-SNE visualisation of feature space

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from src.features.extractor import EEGFeatureExtractor
from src.features.frequency_domain import compute_psd, band_power, FREQUENCY_BANDS

%matplotlib inline
TASK = 'sleep'
SFREQ_MAP = {'focus': 160, 'stress': 128, 'sleep': 100}
SFREQ = SFREQ_MAP[TASK]
CLASS_NAMES = ['Wake', 'N1', 'N2', 'N3', 'REM']

## 1. Load Preprocessed Epochs

In [ ]:
epochs = np.load(f'../data/processed/{TASK}_data.npy')
labels = np.load(f'../data/processed/{TASK}_labels.npy')
print(f'Epochs: {epochs.shape}  |  Labels: {labels.shape}')

## 2. Extract Features

In [ ]:
extractor = EEGFeatureExtractor(sfreq=SFREQ)
X = extractor.transform(epochs)
feat_names = extractor.get_feature_names()

print(f'Feature matrix: {X.shape}')
print(f'First 5 features: {feat_names[:5]}')

## 3. Band Power Per Sleep Stage

In [ ]:
band_means = {band: [] for band in FREQUENCY_BANDS}

for stage in range(5):
    mask = labels == stage
    if mask.sum() == 0:
        for band in FREQUENCY_BANDS:
            band_means[band].append(0)
        continue
    stage_epochs = epochs[mask]
    stage_bp = []
    for ep in stage_epochs:
        freqs, psd = compute_psd(ep, SFREQ)
        ch_mean = psd.mean(axis=0)   # average across channels
        stage_bp.append({b: band_power(ch_mean[np.newaxis], freqs, lo, hi).item()
                          for b, (lo, hi) in FREQUENCY_BANDS.items()})
    for band in FREQUENCY_BANDS:
        band_means[band].append(np.mean([ep[band] for ep in stage_bp]))

# Heatmap
import pandas as pd
df = pd.DataFrame(band_means, index=CLASS_NAMES)
df_norm = df.div(df.max())

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(df_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Relative Band Power per Sleep Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix (Top 30 Features)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X, labels)
top_idx = np.argsort(rf.feature_importances_)[::-1][:30]

corr = np.corrcoef(X[:, top_idx].T)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, xticklabels=[feat_names[i] for i in top_idx],
            yticklabels=[feat_names[i] for i in top_idx],
            cmap='coolwarm', center=0, ax=ax, linewidths=0.3)
ax.set_title('Feature Correlation — Top 30 by Importance', fontsize=12, fontweight='bold')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.show()

## 5. t-SNE Visualisation

In [ ]:
# Use a random subset for speed
rng = np.random.default_rng(42)
n_plot = min(2000, len(X))
idx = rng.choice(len(X), n_plot, replace=False)
X_sub = StandardScaler().fit_transform(X[idx])
y_sub = labels[idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_2d = tsne.fit_transform(X_sub)

colors = ['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']
fig, ax = plt.subplots(figsize=(10, 8))
for cls, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    mask = y_sub == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=name, color=color,
               s=10, alpha=0.6)
ax.set_title('t-SNE of EEG Features by Sleep Stage', fontsize=13, fontweight='bold')
ax.legend(markerscale=3, fontsize=10)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()